# **Import dependencies**

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
from pathlib import Path
import csv
import os
import glob
from pandas.core.groupby.groupby import isna

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Install owlready2
import sys
sys.path.insert(0, '/content/drive/MyDrive/Colab Notebooks/lib/Owlready2')
!pip install Owlready2 -q
from owlready2 import *

# **Configuring paths and files**

In [3]:
# Defining Paths
main_path = Path("/content/drive/MyDrive/Colab Notebooks/files/input")
year = '2021'
name_file_city = "Cities-IBGE.xls"
name_file_biome = "Cities_Biomes.xls"
name_file_climate = "Cities_climate_Koppen.xls"
name_file_code_mapbiomas = "Class_Code_Mapbiomas.xls"
name_file_taxas_emissao_cultivos = "Crop_Emission_Rate.xls"

In [4]:
# Loading input datasets
cities_df = pd.read_excel (main_path / name_file_city,  sheet_name='DTB_2022_Municipio')
biomes_df = pd.read_excel (main_path / name_file_biome,  sheet_name='Planilha1')
climate_df = pd.read_excel (main_path / name_file_climate,  sheet_name='Data')
code_mapbiomas_df = pd.read_excel (main_path / name_file_code_mapbiomas,  sheet_name='Class_Code_Mapbiomas')
taxas_emissao_cultivos_df = pd.read_excel (main_path / name_file_taxas_emissao_cultivos, sheet_name='Emissao')

In [5]:
# Resets dataframe indexes
taxas_emissao_cultivos_df.set_index(['Codigo Municipal IBGE'], drop=True, inplace=True)
cities_df.set_index(['Código Município Completo'], drop=True, inplace=True)
biomes_df.set_index(['CD_GEOCMU'], drop=True, inplace=True)
climate_df.set_index(['IBGE-Code'], drop=True, inplace=True)

In [6]:
# Defining dictionaries
integrated_data_dic = {}
carbon_soil_ha_dic = {}
code_mapbiomas_dict = code_mapbiomas_df.to_dict('records')

In [7]:
# Defining the list of areas where there are CO2 emission values (BRLUC)
list_of_accepted_areas = {1,3,4,5,9,10,12,13,14,15,18,19,20,21,36,39,40,41,46,47,48,49,62}

In [8]:
# Accessing the desired year folder
current_path = main_path / year
current_path

PosixPath('/content/drive/MyDrive/Colab Notebooks/files/input/2021')

# **Generate files totaling by areas**

In [9]:
import os
import pandas as pd
import numpy as np
import unicodedata

biomes_dir = os.listdir(current_path)
biomes_dir = sorted(biomes_dir)

# --- CORREÇÃO AQUI: Filtra para listar APENAS PASTAS (ignora os .csv soltos) ---
biomes_dir = [d for d in os.listdir(current_path) if os.path.isdir(os.path.join(current_path, d))]
biomes_dir = sorted(biomes_dir)

for biome_dir in biomes_dir:
    print('Biome: ' + biome_dir)

    # Cria uma lista vazia para acumular os dados APENAS deste bioma
    biome_data_frames = []

    biome_path = current_path / biome_dir

    for root, subFolder, filename in os.walk(biome_path):
      for folder in subFolder:
        city_cod = int(folder)
        internal_path = biome_path / folder
        file_classes_area = internal_path.glob('*classes_area*.*')
        file_carbon_soil = internal_path.glob('*cos_0_30cm*.*')

        # Reset variables
        integrated_data_dic = {}

        # Recovering soil carbon values in each municipality
        for fileY in file_carbon_soil:
          # definindo o dataframe para carbono no solo
          carbon_soil_city_df = pd.read_csv(internal_path / fileY)
          for indice, linha in carbon_soil_city_df.iterrows():
            value_gt_cos = float(linha['Gt COS'])
            value_ton_cos = value_gt_cos * 1000000000
            value_ton_ha = value_ton_cos / float(linha['Área ha'])
            carbon_soil_ha_dic[linha['class']] = value_ton_ha

          # Retrieving location data, carbon emissions and performing consolidations
          for fileX in file_classes_area:
            # Defining the dataframe for crop classes
            classe_area_df = pd.read_csv (internal_path / fileX)
            # Excluding columns 0
            classe_area_df.drop('0', axis=1, inplace=True)
            # Returning with the column name "Imovel"
            classe_area_df = classe_area_df.rename({"area_Imovel": "Imovel"}, axis='columns')
            # Replacing empty 'nan' values with 0
            classe_area_df.replace(np.nan, 0, inplace = True)
            print('City code: ' + str(city_cod))

            for indice, linha in classe_area_df.iterrows():
                for i in range(1,62):
                    if (linha.iloc[i]> 0.0):
                        if (int (classe_area_df.columns[i]) in list_of_accepted_areas):
                            farm = str(linha['Imovel']).replace('farm_', '').split('.')[0]
                            farm = farm[:-2]
                            key = str(linha['Imovel']) + '_' +  str(i)
                            area_cod = str(i)
                            area_name = code_mapbiomas_dict[int(classe_area_df.columns[i])-1]['Class']
                            area_size = linha.iloc[i]
                            CO_emission_ha = taxas_emissao_cultivos_df[int(classe_area_df.columns[i])][city_cod]
                            CO_emission_area = float(linha.iloc[i]) * (taxas_emissao_cultivos_df[int(classe_area_df.columns[i])][city_cod])
                            CO_stock_ha = carbon_soil_ha_dic[int(classe_area_df.columns[i])]
                            CO_stock_area = float(linha.iloc[i]) * float(carbon_soil_ha_dic[int(classe_area_df.columns[i])])

                            city_name = cities_df["Nome_Município"][city_cod]
                            city_name = city_name.replace(" ", "_")
                            state_name = cities_df["Nome_UF"][city_cod]
                            state_cod = str(city_cod)
                            state_cod = state_cod[:2]
                            biome = biomes_df["BIOMA"][city_cod]

                            match biome:
                                case ('Amazônia'):
                                  biome_cod = 1
                                case ('Caatinga'):
                                  biome_cod = 2
                                case ('Cerrado'):
                                  biome_cod = 3
                                case ('Mata Atlântica'):
                                  biome_cod = 4
                                case ('Pampa'):
                                  biome_cod = 5
                                case ('Pantanal'):
                                  biome_cod = 6
                            climate = climate_df["Köppen"][city_cod]
                            match climate:
                                case ('Af'):
                                  # hot climate, no dry season
                                  climate_cod = 1
                                case ('Am'):
                                  # hot monsoon climate
                                  climate_cod = 2
                                case ('As'):
                                  # warm weather with winter rain
                                  climate_cod = 3
                                case ('Aw'):
                                  # hot weather with summer rain
                                  climate_cod = 4
                                case ('BSh'):
                                  # semi-arid and hot
                                  climate_cod = 5
                                case ('Cfa'):
                                  # temperate climate, no dry season and hot summer
                                  climate_cod = 6
                                case ('Cfb'):
                                  # temperate climate, no dry season and cool summer
                                  climate_cod = 7
                                case ('Cwa'):
                                  # temperate climate with hot and humid summer
                                  climate_cod = 8
                                case ('Cwb'):
                                  # temperate climate with cool, humid summer
                                  climate_cod = 9

                            integrated_data_dic[key] = ({'farm': farm,'area_cod': area_cod, 'area_name': area_name, 'area_size': area_size,
                                                         'CO_emission_ha': CO_emission_ha, 'CO_emission_area': CO_emission_area, 'CO_stock_ha': CO_stock_ha,
                                                         'CO_stock_area': CO_stock_area,
                                                         'city_cod': city_cod, 'city_name': city_name, 'state_cod': state_cod, 'state_name': state_name,
                                                         'biome_cod': biome_cod,'biome_name': biome,'climate_cod': climate_cod,'climate_name': climate, 'year': year})

        # Creating a dataframe from the dictionary
        integrated_data_df = pd.DataFrame.from_dict(integrated_data_dic)

        # Transposing the dataframe to flip rows and columns
        integrated_data_df_transposed = integrated_data_df.T

        # Adiciona o dataframe da cidade à lista do bioma (não salva no disco ainda)
        biome_data_frames.append(integrated_data_df_transposed)

    # Fora do loop de cidades, mas dentro do loop do bioma:
    # Consolida as cidades e salva O ARQUIVO DO BIOMA
    if biome_data_frames:
        final_biome_df = pd.concat(biome_data_frames)

        # Changing the folder for saving files
        os.chdir(main_path)
        os.chdir(year)

        # Formata o nome do arquivo corretamente (ex: "Mata Atlântica" -> "mata_atlantica")
        nome_limpo = unicodedata.normalize('NFKD', biome_dir).encode('ASCII', 'ignore').decode('utf-8').lower().replace(" ", "_")
        filename_out = f'data_integration_all_areas_{nome_limpo}.csv'

        if os.path.exists(filename_out):
            os.remove(filename_out)

        final_biome_df.to_csv(filename_out, index=True, header=True)
        print(f'File: {filename_out} generated successfully!')

    # Updating the path para iterar sobre o próximo bioma corretamente
    current_path = main_path / year








# **Generate files totaling by farm**

---



In [10]:
areas_am_df = pd.read_csv ('/content/drive/MyDrive/Colab Notebooks/files/input/2021/data_integration_areas_areas-am-inference.csv')
areas_am_df

,index,farm,area_cod,area_name,area_size,CO2_emission_ha,CO2_emission_area,CO2_stock_ha,CO2_stock_area,balance_CO2_ha,balance_CO2_area,city_cod,city_name,state_cod,state_name,biome_cod,biome_name,climate_cod,climate_name,year
0,1,farm_4576327,farm_4576327_area_3,Forest Formation,5.35,0.00,0.0000,31.349326,167.718896,31.349326,167.718896,1100908,Castanheiras,11,Rondônia,1,Amazônia,2,Am,2021
1,2,farm_4576327,farm_4576327_area_12,Grassland,0.27,4.64,1.2528,31.344631,8.463050,26.704631,7.210250,1100908,Castanheiras,11,Rondônia,1,Amazônia,2,Am,2021
2,3,farm_4576327,farm_4576327_area_15,Pasture,69.79,4.64,323.8256,29.292698,2044.337396,24.652698,1720.511796,1100908,Castanheiras,11,Rondônia,1,Amazônia,2,Am,2021
3,4,farm_4574927,farm_4574927_area_3,Forest Formation,3.23,0.00,0.0000,31.349326,101.258324,31.349326,101.258324,1100908,Castanheiras,11,Rondônia,1,Amazônia,2,Am,2021
4,5,farm_4574927,farm_4574927_area_12,Grassland,0.30,4.64,1.3920,31.344631,9.403389,26.704631,8.011389,1100908,Castanheiras,11,Rondônia,1,Amazônia,2,Am,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2651,2652,farm_1837206,farm_1837206_area_12,Grassland,1.84,4.64,8.5376,31.344631,57.674122,26.704631,49.136522,1100908,Castanheiras,11,Rondônia,1,Amazônia,2,Am,2021
2652,2653,farm_1837206,farm_1837206_area_15,Pasture,169.45,4.64,786.2480,29.292698,4963.647682,24.652698,4177.399682,1100908,Castanheiras,11,Rondônia,1,Amazônia,2,Am,2021
2653,2654,farm_4131657,farm_4131657_area_3,Forest Formation,0.01,0.00,0.0000,31.349326,0.313493,31.349326,0.313493,1100908,Castanheiras,11,Rondônia,1,Amazônia,2,Am,2021
2654,2655,farm_4131657,farm_4131657_area_12,Grassland,0.55,4.64,2.5520,31.344631,17.239547,26.704631,14.687547,1100908,Castanheiras,11,Rondônia,1,Amazônia,2,Am,2021


In [16]:
import pandas as pd
import os
import glob
from pathlib import Path

# Including in dataframe

# Defining Paths
main_path = Path("/content/drive/MyDrive/Colab Notebooks/files/input")
year = '2021'
current_path = main_path / year
os.chdir(current_path)

# --- ALTERAÇÃO 1: Lendo todos os biomas divididos por áreas dinamicamente ---
# Busca todos os arquivos de bioma na pasta atual
all_filenames = glob.glob('data_integration_all_areas_*.csv')
# Filtra para garantir que não vai ler arquivos de teste ou finais antigos
all_filenames = [f for f in all_filenames if 'final' not in f and 'inference' not in f]

print(f"Lendo os seguintes arquivos: {all_filenames}")

# Concatena todos os biomas em um único dataframe
areas_am_df = pd.concat([pd.read_csv(f) for f in all_filenames])

columns_farms = ['index','farm','total_area_size', 'total_emission_area', 'total_stock_area','balance_CO2_farm', 'balance_CO2_farm_ha', 'city_cod','city_name',
           'state_cod','state_name','biome_cod','biome_name','climate_cod','climate_name','year']
farms_am_df = pd.DataFrame(columns=columns_farms)

# Ordering the dataframe
areas_am_df = areas_am_df.sort_values(by='farm')
# --- ALTERAÇÃO 2: Reset obrigatório do índice após o concat para evitar erro no .at[0] ---
areas_am_df.reset_index(drop=True, inplace=True)

# Defining variables
current_farm = areas_am_df.at[0, 'farm']
city_cod = areas_am_df.at[0, 'city_cod']
city_name = areas_am_df.at[0,'city_name']
#year = areas_am_df.at[0, 'year']
area_size_total = 0.0
# Fix: Changed 'CO2_emission_area_total' to 'CO_emission_area_total' to match the column name in the input CSV files
CO_emission_area_total = 0.0
# Fix: Changed 'CO2_stock_area_total' to 'CO_stock_area_total' to match the column name in the input CSV files
CO_stock_area_total = 0.0
balance_CO2_farm = 0.0
balance_CO2_farm_ha = 0.0
index_farm = 1
key = 1
count_elements = len(areas_am_df)

# Iterating over the lines and performing processing
for index, row in areas_am_df.iterrows():
  farm = row['farm']
  if (farm == current_farm):
    area_size_total = float(area_size_total) + float(row['area_size'])
    # Verifique se a sua coluna é CO_emission_area ou CO2_emission_area e ajuste se necessário
    # Fix: Changed 'CO2_emission_area' to 'CO_emission_area' to use the correct column from the input files
    CO_emission_area_total = float(CO_emission_area_total) + float(row['CO_emission_area'])
    # Fix: Changed 'CO2_stock_area' to 'CO_stock_area' to use the correct column from the input files
    CO_stock_area_total = float(CO_stock_area_total) + float(row['CO_stock_area'])
    current_farm = farm
    count_elements = count_elements -1
  else:
    balance_CO2_farm = CO_stock_area_total - CO_emission_area_total
    if (area_size_total != 0.0):
      balance_CO2_farm_ha = float(balance_CO2_farm) / float(area_size_total)
    farms_am_df.loc[key] = {'index': index_farm,'farm': current_farm.replace('farm_',''), 'total_area_size': area_size_total, 'total_emission_area': CO_emission_area_total, 'total_stock_area': CO_stock_area_total,
               'balance_CO2_farm': balance_CO2_farm, 'balance_CO2_farm_ha': balance_CO2_farm_ha, 'city_cod': areas_am_df.at[index, 'city_cod'],
               'city_name': areas_am_df.at[index,'city_name'],'state_cod': areas_am_df.at[index, 'state_cod'], 'state_name': areas_am_df.at[index,'state_name'],
               'biome_cod': areas_am_df.at[index, 'biome_cod'], 'biome_name': areas_am_df.at[index, 'biome_name'], 'climate_cod': areas_am_df.at[index, 'climate_cod'],
               'climate_name': areas_am_df.at[index, 'climate_name'], 'year': areas_am_df.at[index, 'year']}
    area_size_total = float(row['area_size'])
    # Fix: Changed 'CO_emission_area' to 'CO_emission_area' to use the correct column from the input files
    CO_emission_area_total = float(row['CO_emission_area'])
    # Fix: Changed 'CO_stock_area' to 'CO_stock_area' to use the correct column from the input files
    CO_stock_area_total = float(row['CO_stock_area'])
    current_farm = farm
    index_farm = index_farm + 1
    key = key + 1
    count_elements = count_elements -1
    if (count_elements == 0):
      balance_CO2_farm = CO_stock_area_total - CO_emission_area_total
      if (area_size_total != 0.0):
        balance_CO2_farm_ha = float(balance_CO2_farm) / float(area_size_total)
      farms_am_df.loc[key] = {'index': index_farm, 'farm': current_farm.replace('farm_',''), 'total_area_size': area_size_total, 'total_emission_area': CO_emission_area_total, 'total_stock_area': CO_stock_area_total,
                              'balance_CO2_farm': balance_CO2_farm, 'balance_CO2_farm_ha': balance_CO2_farm_ha, 'city_cod': areas_am_df.at[index, 'city_cod'],
                              'city_name': areas_am_df.at[index,'city_name'],'state_cod': areas_am_df.at[index, 'state_cod'], 'state_name': areas_am_df.at[index,'state_name'],
                              'biome_cod': areas_am_df.at[index, 'biome_cod'], 'biome_name': areas_am_df.at[index, 'biome_name'], 'climate_cod': areas_am_df.at[index, 'climate_cod'],
                              'climate_name': areas_am_df.at[index, 'climate_name'], 'year': areas_am_df.at[index, 'year']}

# Changing the folder for saving files
os.chdir(main_path)
os.chdir(year)

# Generating csv file with integrated data
filename_out = 'data_integration_all_farms_final.csv'
path_out = filename_out

if os.path.exists(path_out):
  os.remove(path_out)

farms_am_df.to_csv(filename_out, index=False, header=True)
print (f'File: {filename_out} successfully generated!')

# Updating the path
current_path = main_path / year

Lendo os seguintes arquivos: ['data_integration_all_areas_amazonia.csv', 'data_integration_all_areas_mata_atlantica.csv', 'data_integration_all_areas_pantanal.csv', 'data_integration_all_areas_cerrado.csv', 'data_integration_all_areas_pampa.csv', 'data_integration_all_areas_caatinga.csv']


KeyError: 'CO2_emission_area'

# **Preparing data for ontology**

In [ ]:
# Accessing the desired year folder
current_path = main_path / year
current_path

# Preparing datasets
biomes_dir = os.listdir(current_path)
biomes_dir = sorted(biomes_dir)
for biome_dir in biomes_dir:
    print('Biome: ' + biome_dir)
    current_path = current_path / biome_dir
    for root, subFolder, filename in os.walk(current_path):
      for folder in subFolder:
        city_cod = int(folder)
        internal_path = current_path / folder
        file_classes_area = internal_path.glob('*classes_area*.*')
        file_carbon_soil = internal_path.glob('*cos_0_30cm*.*')

        # Reset variables
        onto_data_dic = {}

        # Recovering soil carbon values in each municipality
        for fileY in file_carbon_soil:
          # definindo o dataframe para carbono no solo
          carbon_soil_city_df = pd.read_csv(internal_path / fileY)
          for indice, linha in carbon_soil_city_df.iterrows():
            value_gt_cos = float(linha['Gt COS'])
            value_ton_cos = value_gt_cos * 1000000000
            value_ton_ha = value_ton_cos / float(linha['Área ha'])
            carbon_soil_ha_dic[linha['class']]= linha['class']
            carbon_soil_ha_dic[linha['class']] = value_ton_ha

          # Retrieving location data, carbon emissions and performing consolidations
          for fileX in file_classes_area:
            # Defining the dataframe for crop classes
            classe_area_df = pd.read_csv (internal_path / fileX)
            # Excluding columns 0
            classe_area_df.drop('0', axis=1, inplace=True)
            # Reaming the columns
            #classe_area_df = classe_area_df.rename(columns=lambda x: 'area_' + x)
            # Returning with the column name "Imovel"
            classe_area_df = classe_area_df.rename({"area_Imovel": "Imovel"}, axis='columns')
            # Replacing empty 'nan' values with 0
            classe_area_df.replace(np.nan, 0, inplace = True)
            print('City code: ' + str(city_cod))
            for indice, linha in classe_area_df.iterrows():
                for i in range(1,62):
                    if (linha.iloc[i]> 0.0):
                        if (int (classe_area_df.columns[i]) in list_of_accepted_areas):
                            farm = str(linha['Imovel'])
                            farm = farm[:-2]
                            farm = 'farm_' + farm
                            key = str(linha['Imovel']) + '_' +  str(i)
                            area_cod = classe_area_df.columns[i]
                            area_cod = farm + '_area_' + area_cod
                            area_name = code_mapbiomas_dict[int(classe_area_df.columns[i])-1]['Class']
                            area_size = linha.iloc[i]
                            CO_emission_ha = taxas_emissao_cultivos_df[int(classe_area_df.columns[i])][city_cod]
                            #CO_emission_area = float(linha.iloc[i]) * (taxas_emissao_cultivos_df[int(classe_area_df.columns[i])][city_cod])
                            #print (area_name + ' - ' + classe_area_df.columns[i])
                            CO_stock_ha = carbon_soil_ha_dic[int(classe_area_df.columns[i])]
                            #CO_stock_area = float(linha.iloc[i]) * float(carbon_soil_ha_dic[int(classe_area_df.columns[i])])
                            #balance_CO2_area = float(CO_stock_area) - float(CO_emission_area)
                            #balance_CO2_ha = float(balance_CO2_area) / float(area_size)
                            city_name = cities_df["Nome_Município"][city_cod]
                            city_name = city_name.replace(" ", "_")
                            state_name = cities_df["Nome_UF"][city_cod]
                            state_name = state_name.replace(" ", "_")
                            state_cod = str(city_cod)
                            state_cod = state_cod[:2]
                            biome = biomes_df["BIOMA"][city_cod]
                            match biome:
                                case ('Amazônia'):
                                    biome_cod = 1
                                case ('Caatinga'):
                                    biome_cod = 2
                                case ('Cerrado'):
                                    biome_cod = 3
                                case ('Mata_Atlântica'):
                                    biome_cod = 4
                                case ('Pampa'):
                                    biome_cod = 5
                                case ('Pantanal'):
                                    biome_cod = 6
                            climate = climate_df["Köppen"][city_cod]
                            match climate:
                                case ('Af'):
                                    # hot climate, no dry season
                                    climate_cod = 1
                                case ('Am'):
                                    # hot monsoon climate
                                    climate_cod = 2
                                case ('As'):
                                    # warm weather with winter rain
                                    climate_cod = 3
                                case ('Aw'):
                                    # hot weather with summer rain
                                    climate_cod = 4
                                case ('BSh'):
                                    # semi-arid and hot
                                    climate_cod = 5
                                case ('Cfa'):
                                    # temperate climate, no dry season and hot summer
                                    climate_cod = 6
                                case ('Cfb'):
                                    # temperate climate, no dry season and cool summer
                                    climate_cod = 7
                                case ('Cwa'):
                                    # temperate climate with hot and humid summer
                                    climate_cod = 8
                                case ('Cwb'):
                                    # temperate climate with cool, humid summer
                                    climate_cod = 9

                            onto_data_dic[key] = ({'farm': farm,'area_cod': area_cod, 'area_name': area_name, 'area_size': area_size,
                                                    'CO2_emission_ha': CO_emission_ha, 'CO2_stock_ha': CO_stock_ha,'city_cod': city_cod,
                                                    'city_name': city_name, 'state_cod': state_cod, 'state_name': state_name,
                                                    'biome_cod': biome_cod,'biome_name': biome,'climate_cod': climate_cod,'climate_name': climate, 'year': year})

        # Creating a dataframe from the dictionary
        onto_data_df = pd.DataFrame.from_dict(onto_data_dic)

        # Transposing the dataframe to flip rows and columns
        onto_data_df_transposed = onto_data_df.T  # or df1.transpose()
        #print(integrated_data_df_transposed.head(1))

        # Changing the folder for saving files
        os.chdir(main_path)
        os.chdir(year)
        os.chdir(biome_dir)

        # Generating csv file with integrated data
        filename_out = 'data_integration_areas_' + str(city_cod) + '_' + city_name + '_' + year + '.csv'
        path_out = filename_out
        if os.path.exists(path_out):
          print ('Error generating the file: ' + filename_out + '. File already exists!')
        else:
          onto_data_df_transposed.to_csv(filename_out, index=True, header=True)
          print ('File: ' + filename_out + ' successfully generated!')

      # Updating the path
      current_path = main_path / year


**Consolidating datasets into one file**

In [ ]:

# Consolidating datasets into one file

current_path = main_path / year
os.chdir(current_path)

# This prevents you from retrieving test files or farm files.
all_filenames_biomes = [i for i in glob.glob('data_integration_all_areas_*.csv')]
# We remove the 'final' file from the list if it already exists to avoid data duplication.
all_filenames_biomes = [f for f in all_filenames_biomes if 'final' not in f]

print(f"Files to be consolidated: {all_filenames_biomes}")

# Concatenating only the biome files
combined_csv_final = pd.concat([pd.read_csv(f) for f in all_filenames_biomes])
all_areas_final_df = combined_csv_final

if 'index' in all_areas_final_df.columns:
    all_areas_final_df = all_areas_final_df.drop(columns=['index'])

# Guarantee of cleanliness and standardization
all_areas_final_df['farm'] = all_areas_final_df['farm'].astype(str).str.replace('farm_', '').str.split('.').str[0]
if 'area_cod' in all_areas_final_df.columns:
    all_areas_final_df['area_cod'] = all_areas_final_df['area_cod'].astype(str).str.split('_area_').str[-1]

# Creating the connection key
all_areas_final_df['area_id_onto'] = all_areas_final_df['farm'] + "_" + all_areas_final_df['area_cod']

# Merge with the ontology results
print("Integrating pellet calculations into the final file")
onto_path = "/content/drive/MyDrive/Colab Notebooks/files/onto/records_areas_am_onto.csv"

if os.path.exists(onto_path):
    records_onto_df = pd.read_csv(onto_path)

    # Clears the IDs from the ontology file to match the final one (in case they are still coming with 'farm_' or '_area_').
    if 'area_id_onto' in records_onto_df.columns:
        records_onto_df['area_id_onto'] = records_onto_df['area_id_onto'].astype(str).str.replace('farm_', '').str.replace('_area_', '_')
    else:
        # If the first column is the ID but has a different name
        records_onto_df.rename(columns={records_onto_df.columns[0]: 'area_id_onto'}, inplace=True)
        records_onto_df['area_id_onto'] = records_onto_df['area_id_onto'].astype(str).str.replace('farm_', '').str.replace('_area_', '_')

    # Do the Merge
    all_areas_final_df = pd.merge(all_areas_final_df, records_onto_df, on='area_id_onto', how='left')

    # We sorted only by biome, state, and city codes.
    sort_columns = ['biome_cod', 'state_cod', 'city_cod']
    all_areas_final_df = all_areas_final_df.sort_values(by=sort_columns).reset_index(drop=True)

    # Reordering the columns of ontology
    cols_onto = ['CO2_emission_area', 'CO2_stock_area', 'balance_CO2_ha', 'balance_CO2_area']
    existing_onto_cols = [c for c in cols_onto if c in all_areas_final_df.columns]

    if existing_onto_cols:
        other_cols = [c for c in all_areas_final_df.columns if c not in existing_onto_cols]
        target_col = 'CO2_stock_ha' if 'CO2_stock_ha' in other_cols else 'CO_stock_ha'

        if target_col in other_cols:
            idx = other_cols.index(target_col) + 1
            new_order = other_cols[:idx] + existing_onto_cols + other_cols[idx:]
            all_areas_final_df = all_areas_final_df[new_order]

    # Optional: Remove the bridge after the merge.
    # all_areas_final_df.drop(columns=['area_id_onto'], inplace=True)
else:
    print("Error: File records_areas_am_onto.csv not found!")



# Adding the final index
if 'index' in all_areas_final_df.columns:
    all_areas_final_df = all_areas_final_df.drop(columns=['index'])
all_areas_final_df.insert(0, 'index', range(1, 1 + len(all_areas_final_df)))

# Saving to the Output folder
output_final_path = main_path.parent / 'output' / 'final'
os.makedirs(output_final_path, exist_ok=True)
os.chdir(output_final_path)

filename_out = 'data_integration_all_areas_final.csv'
if os.path.exists(filename_out):
    os.remove(filename_out)

all_areas_final_df.to_csv(filename_out, index=False, encoding='utf-8-sig')
print(f' Final file successfully generated in: {output_final_path / filename_out}')

# **Create ontology**

In [ ]:
import os
from owlready2.triplelite import Graph

sqlite_path = "/content/drive/MyDrive/Colab Notebooks/files/onto/quadstore-areas-am.sqlite3"

if os.path.exists(sqlite_path):
    os.remove(sqlite_path)
    print("Old database removed. Starting from scratch.")

if isinstance(onto_path, str):
    onto_path = [onto_path]
# Insert a directory to store ontologies
default_world.set_backend(filename = "/content/drive/MyDrive/Colab Notebooks/files/onto/quadstore-areas-am.sqlite3")

onto_path.append("/content/drive/MyDrive/Colab Notebooks/files/onto")
onto = get_ontology("/content/drive/MyDrive/Colab Notebooks/files/onto/CarbonOnto4-0.owl").load(reload_if_newer = True)


#open bd
#graph = Graph("/content/drive/MyDrive/Colab Notebooks/files/onto/quadstore.sqlite3")
#default_world.set_backend(graph)

# **Populating the ontology**

In [ ]:
# Add the repository for newer versions of OpenJDK.
!add-apt-repository ppa:openjdk-r/ppa -y
!apt-get update

# Install Java 25 (required for version 69.0 of the Jena/Pellet libraries)
!apt-get install openjdk-25-jdk-headless -qq > /dev/null

# Configure the environment variables.
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-25-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-25-openjdk-amd64/bin/java

# Check the version
!java -version

In [ ]:
import csv
import owlready2
from owlready2 import *

# Java memory configuration for Reasoner
owlready2.reasoning.JAVA_MEMORY = 8000

# Create the class map
classes_dict = {c.name: c for c in onto.classes()}

# Open the CSV file
f = open("/content/drive/MyDrive/Colab Notebooks/files/input/2021/data_integration_all_areas_amazonia.csv")
reader = csv.reader(f)
next(reader) # Pula o cabeçalho

# Population stage (Inserting data into memory)
with onto:
    for row in reader:
        if not row: continue

        # Unpacking the columns
        id, farm, area_cod, area_name, area_size, CO_emission_ha, CO_stock_ha, city_cod, city_name, state_cod, state_name, biome_cod, biome_name, climate_cod, climate_name, year = row

        # Cleaning IDs for the ontology
        farm_clean = farm.replace('farm_', '').split('.')[0]
        area_clean = area_cod.split('_area_')[-1] if '_area_' in area_cod else area_cod

        FarmClass     = classes_dict.get("Farm")
        CityClass     = classes_dict.get("City")
        StateClass    = classes_dict.get("State")
        BiomeClass    = classes_dict.get("Biome")
        ClimateClass  = classes_dict.get("Climate")
        FarmAreaClass = classes_dict.get("Farm_Area")

        if not FarmClass:
            print("Error: The class 'Farm' was not found!")
            break

        # Creation of Individuals
        individual  = FarmClass(f"farm_{farm_clean}")
        individual2 = CityClass(city_name.replace(" ", "_"))
        individual3 = StateClass(state_name.replace(" ", "_"))
        individual4 = BiomeClass(biome_name.replace(" ", "_"))
        individual5 = ClimateClass(climate_name)
        individual6 = FarmAreaClass(f"farm_{farm_clean}_area_{area_clean}")

        # Relationships
        individual.isPartOf.append(individual2)
        individual2.isPartOf.append(individual3)
        individual2.isPartOf.append(individual4)
        individual.hasClimate.append(individual5)
        individual.hasPart.append(individual6)
        individual6.hasClimate.append(individual5)

        # Numerical Values
        individual6.hasSizeFarmArea.append(float(area_size))
        individual6.hasEmissionValueCO2FarmAreaHa.append(float(CO_emission_ha))
        individual6.hasSequestrationValueCO2FarmAreaHa.append(float(CO_stock_ha))

  # Reasoning step (Vital: Pellet calculates the balances here)
print("Starting the reasoning with Pellets (this may take a while)")
with onto:
      # This command executes the rules and generates the CO2_Balance values.
      sync_reasoner_pellet(infer_property_values = True, infer_data_property_values = True)

# Stage of extracting real results.

print("Extracting values ​​calculated by the Pellet for Machine Learning")
results_data = []

for area in onto.Farm_Area.instances():
    # We use .first() which returns None instead of throwing an error if the list is empty.
    emission_val = area.hasEmissionValueCO2FarmAreaHa.first() or 0.0
    stock_val    = area.hasSequestrationValueCO2FarmAreaHa.first() or 0.0

    # Safely taking the inferred values
    balance_ha   = area.hasCO2BalanceFarmAreaHa.first() or 0.0
    balance_area = area.hasCO2BalanceFarmArea.first() or 0.0
    size_area    = area.hasSizeFarmArea.first() or 0.0

    # Standardization of the ID: We created the key 'FAZENDA_AREA' (Ex: 1001178_15)
    area_id_clean = area.name.replace('farm_', '').replace('_area_', '_')

    results_data.append({
        'area_id_onto': area_id_clean,
        'balance_CO2_ha': balance_ha,
        'balance_CO2_area': balance_area,
        'CO2_emission_area': (emission_val * size_area),
        'CO2_stock_area': (stock_val * size_area)
    })

# Creating the DataFrame
records_df_am = pd.DataFrame(results_data)

# Saving the CSV
current_path = "/content/drive/MyDrive/Colab Notebooks/files/onto"
os.chdir(current_path)
filename_out = 'records_areas_am_onto.csv'
records_df_am.to_csv(filename_out, index=False, encoding='utf-8-sig')


In [ ]:
import os
from owlready2 import *

# Draft path
sqlite_file = "/content/drive/MyDrive/Colab Notebooks/files/onto/quadstore-inferences-areas-am.sqlite3"

# Preventive cleaning
if os.path.exists(sqlite_file):
    os.remove(sqlite_file)
    print("Old draft removed!")

# Environment configuration
default_world.set_backend(filename = sqlite_file)

# Loading the ontology (we load the file you saved in the previous step)
onto_path = "/content/drive/MyDrive/Colab Notebooks/files/onto/CarbonOnto4-0-individuals-areas-am.owl"
onto_inferences = get_ontology(onto_path).load()

# Pellet Reasoning (Supports SWRL mathematical rules)
with onto_inferences:
    print("Starting the reasoning with Pellets (this may take a while)")
    # infer_data_property_values=True é essencial para salvar os cálculos numéricos
    sync_reasoner_pellet(infer_property_values = True, infer_data_property_values = True)

# Saving the results
output_save = "/content/drive/MyDrive/Colab Notebooks/files/onto/CarbonOnto4-0-inferences-areas-am.owl"
onto_inferences.save(output_save)
print(f"Success! Results calculated and saved in: {output_save}")

# **Generating scripts for ML**

In [ ]:
# # **Generating scripts for ML**

# # Loading the raw data
# areas_am_df = pd.read_csv ('/content/drive/MyDrive/Colab Notebooks/files/input/2021/data_integration_all_areas_amazonia.csv')

# # Cleaning IDs in the original dataframe
# areas_am_df['farm'] = areas_am_df['farm'].astype(str).str.replace('farm_', '').str.split('.').str[0]
# if 'area_cod' in areas_am_df.columns:
#     areas_am_df['area_cod'] = areas_am_df['area_cod'].astype(str).str.split('_area_').str[-1]

# # Created the binding key in the main DataFrame.
# areas_am_df['area_id_onto'] = areas_am_df['farm'] + "_" + areas_am_df['area_cod']

# # Loading inference records
# records_inference_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/files/onto/records_areas_am_onto.csv')

# # Protection against key errors
# # If the column does not exist in the ontology file, we rename the first column.
# if 'area_id_onto' not in records_inference_df.columns:
#     print("Warning: Column 'area_id_onto' not found in the Ontology file. Adjusting automatically")
#     # Assume that the first column of the inference file is the ID we need.
#     records_inference_df.rename(columns={records_inference_df.columns[0]: 'area_id_onto'}, inplace=True)

# # Intelligent integration (merge)
# # We have ensured that 'area_id_onto' exists in both.
# areas_am_df = pd.merge(areas_am_df, records_inference_df, on='area_id_onto', how='left')

# # Removing the temporary column after use
# areas_am_df.drop(columns=['area_id_onto'], inplace=True)

# # Maintaining your original views
# for index, row in areas_am_df.iterrows():
#     pass

# # Saving the final outcome of the biome
# current_path = ('/content/drive/MyDrive/Colab Notebooks/files/input/2021')
# os.chdir(current_path)
# filename_out = 'data_integration_areas-am-inference-teste.csv'

# if os.path.exists(filename_out):
#   os.remove(filename_out)

# areas_am_df.to_csv(filename_out, index=False, header=True)
# print (f'Success! Archive {filename_out} generated using integrated ontology data.')